# B2-022 — Practice p11

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** gaussian-reparameterization

## Task

Implement `reparameterize(mu, logvar, eps)` and verify that gradients reach `mu` and `logvar`.

**Contract.** `mu`, `logvar`, and `eps` are float32 tensors of the same shape `(B,k)`; return `z = mu + torch.exp(0.5 * logvar) * eps` with the same shape and dtype. Raise `ValueError` if the three shapes are not identical.

**Banned inside `reparameterize` (zero points if used):** drawing random numbers (`torch.randn`, `torch.normal`, `.sample()`, `.rsample()`, generators), `torch.distributions`, `.detach()`, `.item()`, `.data`, and in-place modification of any input.

**Probes** with the literals below (`mu` and `logvar` have `requires_grad=True`; `eps` does not; `ATOL = RTOL = 1e-6`):

1. `z` equals `[[1.7, -1.2, 0.0], [1.6321206, 0.5, 2.0680509]]`.
2. After `z.sum().backward()`: `mu.grad` is all ones and `logvar.grad` equals `[[0.6, -0.1, 0.0], [-0.1839397, 0.25, 1.2840254]]`; `eps.grad is None`.
3. With fresh leaf copies and the weighted loss `(w * z).sum()` for the literal `w`: `mu.grad` equals `w`, and `logvar.grad` equals `w * 0.5 * torch.exp(0.5 * logvar) * eps`.
4. **Distribution check.** With `g = torch.Generator().manual_seed(SEED)`, `big_eps = torch.randn(100_000, 3, generator=g)`, and `mu_row`, `logvar_row` expanded to `(100_000, 3)`, the sample mean of `z` is within `0.02` of `mu_row` and the per-coordinate sample variance is within `0.05` of `logvar_row.exp()` (absolute tolerances). Show that the variant using `torch.exp(logvar)` as the standard deviation fails this check.
5. Mismatched shapes `(2,3)`, `(2,3)`, `(3,3)` raise `ValueError`.

**Required answer-check assertions.** Probes 1–5.

In [ ]:
import math
import torch

SEED = 20260927
mu = torch.tensor([[0.5, -1.0, 0.0], [2.0, 0.0, -0.5]], requires_grad=True)
logvar = torch.tensor([[0.0, math.log(0.25), 1.0], [-2.0, 0.0, 0.5]], requires_grad=True)
eps = torch.tensor([[1.2, -0.4, 0.0], [-1.0, 0.5, 2.0]])
w = torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.5, 2.0]])
mu_row = torch.tensor([0.5, -1.0, 0.0])
logvar_row = torch.tensor([0.0, math.log(0.25), 1.0])
ATOL = 1e-6
RTOL = 1e-6

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.